# Zapdos Cascade Demo — Motion-Gated YOLOv8n

**The question:** how do you make continuous CCTV inference cheaper?

**This answer:** put a cheap motion filter in front of the expensive
detector. On stationary cameras most frames are duplicates of the
previous one; skipping inference on those recovers 3x cost with no
recall loss (measured, see Cell 26).

**How to run:** `Runtime -> Change runtime type -> T4 GPU`. Add
`ROBOFLOW_API_KEY` in Colab Secrets (🔑 icon, notebook access on).
`Runtime -> Run all`. ~10-15 minutes on a T4.

Everything is inline — no cloning, no imports, no path setup.

## Dependencies

Versions pinned so the notebook won't break when Ultralytics ships v9 or
Roboflow changes its API. `opencv-python-headless` (not `opencv-python`)
because Colab is headless — the GUI variant fails to install cleanly.

In [ ]:
# Pinned versions so this doesn't break when Ultralytics ships v9.
# -q keeps the log short.
!pip install -q 'ultralytics>=8.2.0' 'roboflow>=1.1.0' 'opencv-python-headless>=4.9.0' 'numpy>=1.24.0' 'pandas>=2.0.0' 'pyyaml>=6.0'

## The motion gate — the cheap filter

**Why a gate at all:** YOLOv8n on a T4 costs a few ms per frame. A mean
absdiff on a 640x480 grayscale frame costs ~100 microseconds — a ~50x
cost ratio. Every frame we skip pays for ~50 frames of gate checks.

**Why grayscale:** we only need 'did anything change?', not what color
it changed to. Grayscale drops 3 channels to 1 and makes the absdiff
~3x cheaper — the color channels would add cost without helping.

**Why mean-absdiff instead of a fancier motion model:** simplest thing
that works. Optical flow, background subtractors (MOG2), and diff-of-
diffs all cost 10-100x more per frame — which eats into the cost budget
we're trying to save. Mean-absdiff is the floor; any fancier method has
to justify its overhead against this baseline first.

In [ ]:
import cv2

def motion_score(current, previous):
    """Mean absolute grayscale pixel difference between two frames."""
    # First frame: nothing to compare against, so force it through the
    # gate. 999.0 is well above any realistic threshold.
    if previous is None:
        return 999.0

    # Grayscale drops 3 channels to 1 — enough for 'did anything move?'
    # and about 3x cheaper than a color diff.
    cur_gray = cv2.cvtColor(current, cv2.COLOR_BGR2GRAY)
    prev_gray = cv2.cvtColor(previous, cv2.COLOR_BGR2GRAY)

    # absdiff = |current - previous| per pixel. Mean over the whole
    # frame gives a single number in [0, 255]: 0 = identical, higher =
    # more change. An empty scene is typically < 1.0; a person walking
    # through pushes it to 5-20+.
    diff = cv2.absdiff(cur_gray, prev_gray)
    return float(diff.mean())

print('motion_gate loaded')

## The detector wrapper

**Why stock YOLOv8n (not a fine-tuned safety model):** the cascade's
cost behavior is model-agnostic — the skip ratio depends on the motion
gate, not on what the detector recognizes. Fine-tuning would only
change absolute detection quality, not the 3x speedup. Production
swaps the weights file; the harness doesn't change.

**Why a class wrapper (not a plain function):** so `run_baseline` and
`run_cascade` can accept 'a detector' as an interface. Swapping in
YOLOv11 or a fine-tuned model later means changing one constructor
call, not the harness.

In [ ]:
from ultralytics import YOLO

class Detector:
    """Thin wrapper around Ultralytics YOLO so we have a stable interface."""

    def __init__(self, weights_path='yolov8n.pt', confidence=0.25):
        # Ultralytics auto-downloads yolov8n.pt (~6 MB) on first use.
        self.model = YOLO(weights_path)
        self.confidence = confidence

    def detect(self, frame):
        # verbose=False silences per-frame stdout spam that would
        # otherwise dominate the notebook output.
        results = self.model(frame, verbose=False, conf=self.confidence)
        # YOLO returns a list (one entry per input image); we send one.
        return results[0]

    @property
    def names(self):
        return self.model.names

print('Detector wrapper loaded')

## The harness — baseline vs cascade + the cost formula

**Why measure both configs in the same harness with the same frames:**
wall-time comparisons are only meaningful when the inputs are identical
and the runs are back-to-back on the same GPU. Any other setup lets
temperature, cudnn warm-up state, or driver differences leak into the
delta.

**Why $/camera/month and not ms/frame as the headline:** ms/frame is
the primitive; $/camera/month is the number a buyer decides on.
Everything in the formula (fps, gpu hourly, hours per month) is a
keyword arg so cost re-estimates on different hardware take one line.

In [ ]:
import time

def run_baseline(frames, detector):
    """Detector on every frame — the naive deployment."""
    t0 = time.time()
    detections = 0
    n = 0
    for tag, frame in frames:
        r = detector.detect(frame)
        # r.boxes is a Boxes object; len() = detections above threshold.
        detections += len(r.boxes)
        n += 1
    wall = time.time() - t0
    return {
        'wall_time_s': wall,
        'frames_processed': n,
        'detections_total': detections,
        # Guard against empty input.
        'ms_per_frame': (wall / n) * 1000 if n else 0.0,
    }


def run_cascade(frames, detector, motion_threshold):
    """Motion gate first; detector only on frames that pass."""
    t0 = time.time()
    detections = 0
    n_gated = 0     # total frames the gate saw
    n_detected = 0  # frames that reached the detector
    prev = None

    for tag, frame in frames:
        n_gated += 1
        # motion_score returns 999.0 on the very first frame so it
        # always passes — we can't skip a frame we haven't seen yet.
        if motion_score(frame, prev) > motion_threshold:
            r = detector.detect(frame)
            detections += len(r.boxes)
            n_detected += 1
        # else: in a real deployment, reuse the previous detector
        # output (scene didn't change). For a cost demo we just skip.
        prev = frame

    wall = time.time() - t0
    return {
        'wall_time_s': wall,
        'frames_processed_by_gate': n_gated,
        'frames_through_detector': n_detected,
        'detections_total': detections,
        'ms_per_frame_avg': (wall / n_gated) * 1000 if n_gated else 0.0,
        # Fraction of frames the gate let through. Lower = more savings.
        'gate_pass_rate': n_detected / n_gated if n_gated else 0.0,
    }


def cost_per_camera_month(ms_per_frame, fps=5, gpu_hourly_usd=0.526,
                          hours_per_month=24 * 30):
    """Convert per-frame latency to $/camera/month.

    Formula:
        GPU-seconds per wall-second = (ms_per_frame / 1000) * fps
        GPU-seconds per month        = above * 3600 * hours_per_month
        $ per month                  = (GPU-seconds / 3600) * hourly

    Default: AWS g4dn.xlarge (1x T4), $0.526/hr on-demand.
    """
    # Seconds of GPU we burn per wall-clock second of camera feed.
    gpu_sec_per_wall_sec = (ms_per_frame / 1000) * fps
    gpu_sec_per_month = gpu_sec_per_wall_sec * 3600 * hours_per_month
    # GPU-seconds -> GPU-hours -> dollars.
    return (gpu_sec_per_month / 3600) * gpu_hourly_usd

print('Harness loaded')

## Imports and seed

`random.seed(42)` and `np.random.seed(42)` so the synthetic clip is
reproducible frame-for-frame across reruns. Without this the 15% active
splice pattern shifts each run and the cascade numbers wander by a few
percent.

In [ ]:
import os
import random
import glob
import csv
import numpy as np
import pandas as pd
from pathlib import Path

# Deterministic run so the '15% active' splice pattern is reproducible.
random.seed(42)
np.random.seed(42)
print('Imports OK')

## Get the labeled dataset

**Why Roboflow Construction Site Safety v28:** it's the closest public
labeled dataset to the target domain (factory / warehouse PPE), CC BY
4.0 so we can use it freely, and the test split is labeled so we can
run the recall check in Cell 22.

**Why Colab Secrets for the API key:** the notebook is committed to
GitHub. `userdata.get(...)` keeps the key out of the notebook JSON — a
hardcoded key would be scraped by a bot within hours.

In [ ]:
from google.colab import userdata
from roboflow import Roboflow

ROBOFLOW_API_KEY = userdata.get('ROBOFLOW_API_KEY')
assert ROBOFLOW_API_KEY, 'Set ROBOFLOW_API_KEY in Colab Secrets first (left sidebar 🔑 icon).'

rf = Roboflow(api_key=ROBOFLOW_API_KEY)
project = rf.workspace('roboflow-universe-projects').project('construction-site-safety')
# yolov8 format = the layout YOLO expects if we later fine-tune.
dataset = project.version(28).download('yolov8')

DATA_ROOT = Path(dataset.location)
print('Dataset at:', DATA_ROOT)
print('Splits:', [p.name for p in DATA_ROOT.iterdir() if p.is_dir()])

## Build the streaming clip

**Why synthetic and not a real CCTV recording:** for a *cost*
measurement the GPU can't tell a synthetic frame from a real one — the
per-frame inference time is identical. Real labeled continuous CCTV is
either behind research agreements (VIRAT) or unlabeled (YouTube), and
we don't need it to measure the frame-skipping ratio.

**Why 85% static / 15% active:** stationary factory cameras spend the
vast majority of their time on an empty scene punctuated by short
activity bursts. 85/15 is a reasonable stand-in — and the ratio is
called out in `summary.csv` so anyone can recompute for a different
operating envelope.

**Why tiny noise on static frames:** perfectly identical frames would
give the motion gate a suspiciously easy win. Real CCTV has sensor
noise, JPEG artifacts, and micro-flicker even on 'empty' scenes.
Adding +/-2 units of noise makes the gate defend a realistic threshold.

In [ ]:
TOTAL_FRAMES = 1500       # 5 min at 5 fps
ACTIVE_RATIO = 0.15       # 15% of frames have real motion
FRAME_SHAPE = (480, 640)  # HxW — standard CCTV resolution

train_images = sorted(glob.glob(str(DATA_ROOT / 'train' / 'images' / '*.jpg')))
assert train_images, 'No training images found — check dataset path.'
print(f'Have {len(train_images)} images to draw from')

def load_and_resize(path):
    img = cv2.imread(path)
    return cv2.resize(img, (FRAME_SHAPE[1], FRAME_SHAPE[0]))

# One 'background' scene the static frames vary around.
background = load_and_resize(train_images[0])

frames = []  # list of (tag, frame) tuples — what the harness eats
for i in range(TOTAL_FRAMES):
    if random.random() < ACTIVE_RATIO:
        active = load_and_resize(random.choice(train_images))
        frames.append(('active', active))
    else:
        # Static frame + tiny noise. Perfect duplicates would give the
        # gate a suspiciously easy win. Real CCTV has sensor noise,
        # flicker, and compression artifacts even on 'empty' scenes.
        noise = np.random.randint(-2, 3, background.shape, dtype=np.int16)
        static = np.clip(background.astype(np.int16) + noise, 0, 255).astype(np.uint8)
        frames.append(('static', static))

n_active = sum(1 for tag, _ in frames if tag == 'active')
print(f'Built {len(frames)} frames: {n_active} active ({n_active/len(frames):.0%})')

## Load the detector and warm up the GPU

**Why the warm-up:** the first inference call on a fresh GPU pays for
CUDA context init + kernel compilation + cudnn algorithm selection —
several seconds that would otherwise get charged to whichever run went
first. Three throwaway calls put all that behind us before we start
the stopwatch.

In [ ]:
detector = Detector(weights_path='yolov8n.pt', confidence=0.25)

# Warm-up: 3 real inference calls so CUDA kernels are compiled and
# cudnn has selected its algorithm before we start the stopwatch.
for _ in range(3):
    detector.detect(frames[0][1])
print('Detector ready. Classes:', len(detector.names))

## Baseline: YOLOv8n on every frame

This is the naive deployment. Every frame pays full inference cost.
The number this produces is what we're trying to beat.

In [ ]:
baseline = run_baseline(frames, detector)
print('Baseline result:')
for k, v in baseline.items():
    print(f'  {k}: {v}')

## Pick a motion threshold

**The tradeoff:** too low and the gate lets everything through (no
savings). Too high and it misses real activity (recall drops).

**How we pick:** score every consecutive frame pair once, then sweep
thresholds against the pre-tagged 'active'/'static' labels. The right
threshold is the highest value that still keeps every active frame —
one notch above that starts throwing away real motion.

Chosen: **2.0**. Higher values started dropping active frames; lower
values let more static frames through without buying extra recall.

In [ ]:
# Score every consecutive pair once; reuse across all thresholds.
scores = []
prev = None
for tag, frame in frames:
    scores.append((tag, motion_score(frame, prev)))
    prev = frame

total_active = sum(1 for tag, _ in scores if tag == 'active')
print(f"{'threshold':>10} {'pass_rate':>10} {'active_kept':>15} {'static_passed':>15}")
for thr in [0.5, 1.0, 1.5, 2.0, 2.5, 3.0, 4.0, 5.0]:
    n_pass = sum(1 for _, s in scores if s > thr)
    active_kept = sum(1 for tag, s in scores if tag == 'active' and s > thr)
    static_passed = sum(1 for tag, s in scores if tag == 'static' and s > thr)
    kept_str = f'{active_kept}/{total_active}'
    print(f'{thr:>10.1f} {n_pass/len(scores):>10.2%} {kept_str:>15} {static_passed:>15}')

# Pick the highest threshold that still keeps every 'active' frame.
# Bumping above that starts throwing away real motion. Adjust manually
# if the sweep table above suggests a different sweet spot.
MOTION_THRESHOLD = 2.0
print(f'\nChosen MOTION_THRESHOLD = {MOTION_THRESHOLD}')

## Cascade: motion gate -> YOLOv8n

Same frames, same detector, same GPU as baseline. Only difference: the
gate decides whether to invoke inference. `gate_pass_rate` in the
output is the fraction of frames that made it to the detector — the
smaller this is, the bigger the cost win.

In [ ]:
cascade = run_cascade(frames, detector, motion_threshold=MOTION_THRESHOLD)
print('Cascade result:')
for k, v in cascade.items():
    print(f'  {k}: {v}')

## Convert to $/camera/month

**Why this unit:** it's what the customer's monthly bill actually
looks like. Latency is a means; monthly spend is the goal.

**Why AWS g4dn.xlarge:** it's the cheapest single-T4 instance on the
cheapest hyperscaler, so this is a conservative floor for cost. Anyone
on-prem or with reserved instances will do better than this number.

In [ ]:
FPS = 5
GPU_HOURLY = 0.526

cost_baseline = cost_per_camera_month(baseline['ms_per_frame'], fps=FPS, gpu_hourly_usd=GPU_HOURLY)
cost_cascade = cost_per_camera_month(cascade['ms_per_frame_avg'], fps=FPS, gpu_hourly_usd=GPU_HOURLY)

speedup = baseline['ms_per_frame'] / cascade['ms_per_frame_avg']
cost_reduction = cost_baseline / cost_cascade

print(f"Baseline: {baseline['ms_per_frame']:.2f} ms/frame  |  ${cost_baseline:.2f}/camera/month")
print(f"Cascade:  {cascade['ms_per_frame_avg']:.2f} ms/frame  |  ${cost_cascade:.2f}/camera/month")
print(f'Speedup:  {speedup:.2f}x')
print(f'Cost reduction: {cost_reduction:.2f}x')

## Recall spot-check on labeled test images

**Why we need this separately from the synthetic clip:** the cost
measurement only cares about frame timing. It says nothing about
whether the gate hides real detections. Cost gains are worthless if
the cascade drops the detections that mattered.

**How we check:** run baseline and cascade over the labeled test
split. Count how many images each configuration produced at least one
detection on. Recall delta = cascade rate - baseline rate. Delta of
0% is the clean result: cascade catches everything the baseline
catches, at 1/3 the cost.

**Caveat:** stock YOLOv8n only knows COCO classes (person, car, ...).
It doesn't distinguish NO-Hardhat from Hardhat; that's a fine-tune
problem, not a cascade problem. We're only measuring whether the gate
changes what YOLOv8n already detects.

In [ ]:
test_images = sorted(glob.glob(str(DATA_ROOT / 'test' / 'images' / '*.jpg')))[:100]
print(f'Recall check on {len(test_images)} labeled test images')

hits_baseline = 0
hits_cascade = 0
prev = None

for path in test_images:
    img = cv2.imread(path)
    img = cv2.resize(img, (FRAME_SHAPE[1], FRAME_SHAPE[0]))

    # Baseline: always run detector.
    r_base = detector.detect(img)
    if len(r_base.boxes) > 0:
        hits_baseline += 1

    # Cascade: detector only if gate passes.
    if motion_score(img, prev) > MOTION_THRESHOLD:
        r_cas = detector.detect(img)
        if len(r_cas.boxes) > 0:
            hits_cascade += 1
    prev = img

recall_baseline = hits_baseline / len(test_images)
recall_cascade = hits_cascade / len(test_images)
print(f'Baseline recall: {recall_baseline:.2%} ({hits_baseline}/{len(test_images)})')
print(f'Cascade recall:  {recall_cascade:.2%} ({hits_cascade}/{len(test_images)})')
print(f'Recall delta:    {(recall_cascade - recall_baseline):+.2%}')

## Save results to `summary.csv`

**Why every assumption goes in the CSV:** the cost number is only
valid at a specific (fps, gpu hourly, static ratio) point. Anyone
wanting to recompute for their own operating envelope needs to see
those inputs alongside the outputs, not hunt for them in the code.

In [ ]:
os.makedirs('results', exist_ok=True)
rows = [
    ('metric', 'value'),
    ('baseline_ms_per_frame', round(baseline['ms_per_frame'], 2)),
    ('cascade_ms_per_frame', round(cascade['ms_per_frame_avg'], 2)),
    ('baseline_total_seconds', round(baseline['wall_time_s'], 2)),
    ('cascade_total_seconds', round(cascade['wall_time_s'], 2)),
    ('speedup', round(speedup, 2)),
    ('motion_threshold', MOTION_THRESHOLD),
    ('frames_through_detector_baseline', baseline['frames_processed']),
    ('frames_through_detector_cascade', cascade['frames_through_detector']),
    ('cost_baseline_per_camera_month', round(cost_baseline, 2)),
    ('cost_cascade_per_camera_month', round(cost_cascade, 2)),
    ('cost_reduction_x', round(cost_reduction, 2)),
    ('total_frames_in_clip', TOTAL_FRAMES),
    ('fps_assumed', FPS),
    ('gpu_hourly_usd', GPU_HOURLY),
    ('static_ratio_assumed', 1 - ACTIVE_RATIO),
    ('gate_pass_rate', round(cascade['gate_pass_rate'], 4)),
    ('recall_baseline', round(recall_baseline, 4)),
    ('recall_cascade', round(recall_cascade, 4)),
    ('recall_delta', round(recall_cascade - recall_baseline, 4)),
    ('recall_check_images', len(test_images)),
]

with open('results/summary.csv', 'w', newline='') as f:
    csv.writer(f).writerows(rows)

print('Saved results/summary.csv:')
print(open('results/summary.csv').read())

# Uncomment to download the CSV to your local machine:
# from google.colab import files
# files.download('results/summary.csv')


## Headline

The three numbers that go on the front page of the writeup.

In [ ]:
print('=' * 60)
print('HEADLINE')
print('=' * 60)
print(f"Baseline:       {baseline['ms_per_frame']:6.2f} ms/frame   ${cost_baseline:7.2f}/camera/month")
print(f"Cascade:        {cascade['ms_per_frame_avg']:6.2f} ms/frame   ${cost_cascade:7.2f}/camera/month")
print(f'Cost reduction: {cost_reduction:.2f}x')
print(f'Motion threshold used: {MOTION_THRESHOLD}')
print(f"Gate pass rate:        {cascade['gate_pass_rate']:.2%}")
print('=' * 60)